In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.nn import functional as F



In [3]:
gpt2 = tiktoken.get_encoding("gpt2")
dataset_json = "dataset.json" #Ruta al json con el dataset

encoder = tiktoken.Encoding(
    name = "encoder",
    pat_str = gpt2._pat_str,
    mergeable_ranks = gpt2._mergeable_ranks,
    special_tokens = {
        **gpt2._special_tokens,
        "<START>": len(gpt2._mergeable_ranks) +1,
        "<END>": len(gpt2._mergeable_ranks) +2,
        "<PAD>": len(gpt2._mergeable_ranks) +3
    }
)

In [10]:
CONTEXT_LENGTH = 300

dataset = pd.read_json(dataset_json)

df_train,df_val = train_test_split(dataset,test_size=0.2,random_state=42,shuffle=True)

# Dataset
class Dataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.X = self.data['en']
        self.y = self.data['es']
        

    def __len__(self):
        return len(self.data)
    
    def adjust_length(self,text):
        pad = [encoder._special_tokens["<PAD>"] for _ in range(CONTEXT_LENGTH-len(text))]
        return np.concatenate((text,pad))
    

    def __getitem__(self, idx):

        en_text = encoder.encode(self.X.iloc[idx])
        es_text = encoder.encode(self.y.iloc[idx])


        #Truncar entrada en caso de que se supere dimension entrada x_n
        if len(en_text) > CONTEXT_LENGTH:
            en_text =  en_text[:300]
        
        if len(es_text) > CONTEXT_LENGTH:
            es_text = es_text[:(CONTEXT_LENGTH-2)]


        en_text = self.adjust_length(en_text)
        es_text = self.adjust_length(es_text)

       
        #Añadimos tokens especiales al targe
        es_text = np.concatenate(([encoder._special_tokens["<START>"]],
                                  es_text[:(CONTEXT_LENGTH-2)],
                                  [encoder._special_tokens["<END>"]]),
                                  axis=0)


        #COnvertimos a tensores
        es_text =  torch.tensor(es_text,dtype=torch.long)
        en_text = torch.tensor(en_text,dtype= torch.long)

        return en_text, es_text

# Crear dataset y dataloader
train_dataset = Dataset(df_train)
train_dataloader = DataLoader(train_dataset, batch_size=32,shuffle=True)

for X_batch, y_batch in train_dataloader:
    print(X_batch)
    print(y_batch)
    break

tensor([[   12,   818,   262,  ..., 50259, 50259, 50259],
        [ 3198,    25,  9175,  ..., 50259, 50259, 50259],
        [36981, 35876,    11,  ..., 50259, 50259, 50259],
        ...,
        [ 1870,  3511,    30,  ..., 50259, 50259, 50259],
        [ 1639,  1183,   307,  ..., 50259, 50259, 50259],
        [ 1639,  1183,  5875,  ..., 50259, 50259, 50259]])
tensor([[50257,  2949,  1556,  ..., 50259, 50259, 50258],
        [50257,    44,   415,  ..., 50259, 50259, 50258],
        [50257,    56,    64,  ..., 50259, 50259, 50258],
        ...,
        [50257,    12,  1587,  ..., 50259, 50259, 50258],
        [50257,    36,  7364,  ..., 50259, 50259, 50258],
        [50257,  5960, 19944,  ..., 50259, 50259, 50258]])


In [7]:
#Encoder
D_EMBEDDING = 512

class Encoder(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        """
        Embedding: Tabla E de dimension VOCAB_SIZE x D_EMBEDDING
        x_n tendrá dimension D_EMBEDDING
        Pytorch no multiplica matrices aquí, solo indica, dado un vector de indices de tokens, que filas de la matriz E extraer
        """
       
        self.token_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D_EMBEDDING, padding_idx= encoder._special_tokens["<PAD>"])

        """
        Positional Encoding: x_n = x_n + r_n -> r_ tiene dimensión D_EMBEDDING
        El objetivo es que permita determinar en que posición se encuentra cada palabra, por eso el numero de embeddings es CONTEXT_LENGT
        """
        self.position_embedding_table = nn.Embedding(num_embeddings= CONTEXT_LENGTH, embedding_dim=D_EMBEDDING)


    

In [ ]:
#Transformer

#Attention Head
"""ARQUITECTURA TRANSFORMER"""

class AttentionHead(nn.Module):
    """Una sola capa de self-attention"""

    def __init__(self, dimension, attention_mask = None):
        super().__init__()
        self.key = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.query = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.value = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias = False)

        self.mask = attention_mask 


        #self.dropout = nn.Dropout(DROPOUT)

    def forward(self,x):

        #B = numero de Batch; T = numero de "tokens"; C = dimension de cada token
        B, T, C = x.shape

        K = self.key(x) # (B, T , C)
        Q = self.key(x) #(B,T,C)
        V = self.key(x) # (B,T,C)

        #Calculamos la Atención QxK^T
        """
        Calculamos la Atencino Q x K^t
        Como tenemos batches, usamos el operador @ que aplica la multiplicacion en las dos ultimas dimensiones B veces
        K.transpose significa transponer la penultima dimensino T con la pultima dimension C. 
        """
        attention = Q @ K.transpose(-2,-1) #(B,T,C) x (B,C,T) = (B,T,T)
        attention = attention /(C ** 0.5)
        attention = attention + self.mask  #Sumamos la mascara para evitar que se fije en tokens <PAD>

        attention = F.s




In [12]:
# Crear dataset y dataloader
train_dataset = Dataset(df_train)
train_dataloader = DataLoader(train_dataset, batch_size=32,shuffle=True)

for X_batch, y_batch in train_dataloader:
    
    print(X_batch.shape)
    break

torch.Size([32, 300])
